# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AkMebo/ML_Lesson1_flyrankAI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row in the primary performance dataset represents the **daily performance metrics for a specific content page belonging to a specific client on a given day**, which can be enriched with its static properties by joining our dimension table.

### Dimension Table: `dim_content`
* **Primary Key (Grain)**: `(client_hash_id, content_hash_id)`
* **Description**: Contains structural catalog metadata for the content pages (e.g., character counts, target keywords, SEO metrics, and creator/generation model details).
* **Timeline**: The date spans from October 2024 to July 2026

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [13]:
import pandas as pd
import duckdb
from google.colab import userdata

try:
    # Retrieve your token securely from the Colab secrets manager
    hf_token = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    raise ValueError("Please add 'HF_TOKEN' with your Hugging Face token in the Colab Secrets manager (key icon 🔑 on the left) and grant notebook access.")

con = duckdb.connect()

# Configure DuckDB secret securely using the loaded token variable
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

# Let's inspect a few sample rows to understand who/what the dim_content table is about
sample_df = con.sql(f"SELECT * FROM read_parquet('{rel}/dim_content.parquet') LIMIT 5").df()
# Use DESCRIBE on the parquet files to see the columns and their data types for dim_content
res = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/dim_content.parquet')")
display(res)

# Let's count total rows, distinct client IDs, and distinct content IDs for the dimension table
stats_query = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT client_hash_id) as unique_clients,
    COUNT(DISTINCT content_hash_id) as unique_contents,
     MIN(content_created_date) as start_date,
     MAX(content_created_date) as end_date
FROM read_parquet('{rel}/dim_content.parquet')
"""
display(con.sql(stats_query).df())

# Let's verify the unique grain (Primary Key) of dim_content
grain_query = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT (client_hash_id || '-' || content_hash_id)) as unique_grain_keys
FROM read_parquet('{rel}/dim_content.parquet')
"""
display(con.sql(grain_query).df())

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ YES     │ NULL    │ 

,total_rows,unique_clients,unique_contents,start_date,end_date
0,519606,84,519606,2024-10-16,2026-07-06


,total_rows,unique_grain_keys
0,519606,519606


## 2. Fields: feature / label / context / excluded

We categorize key columns from our fact table (`fact_content_daily_performance`) and dimension table (`dim_content`) as follows:

### A. Features (Predictors)
* **Content Properties (`dim_content`)**:
  * `char_count`, `word_count` (Content length)
  * `content_type` (Type of asset)
  * `provider_used`, `model_used` (Which AI generated the page)
* **Keyword Context (`dim_content`)**:
  * `search_volume` (Target keyword interest metric)
  * `competition`, `competition_level`, `cpc` (Keyword competitiveness value)
  * `main_intent` (User search intent behind keyword)

### B. Labels (Target Variables to Predict)
* **Performance Metrics (`fact_content_daily_performance`)**:
  * `impressions`, `clicks`, `conversions` (Volumetric outcomes)
  * `position_average` (Search Engine rankings metric)

### C. Context (Metadata / Keys for aggregation & joining)
* `report_date` (Time progression index)
* `client_hash_id`, `content_hash_id` (Entity identifiers)
* `keyword_hash_id`, `url_hash_id` (Asset associations)

### D. Excluded
* `is_deleted` (Boolean flag: We filter out deleted content immediately rather than training models on it)
* `is_published` (Boolean flag: Must be `True` to analyze actual performance metrics; unpublished contents produce zero activity)

In [14]:
# Execute a quick summary to show the split profiles
res_fact = con.sql(f"""
    SELECT
        'fact_content_daily_performance' AS tbl,
        COUNT(*) as total_rows,
        COUNT(columns(*)) as total_cells
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").df()

res_dim = con.sql(f"""
    SELECT
        'dim_content' AS tbl,
        COUNT(*) as total_rows,
        COUNT(columns(*)) as total_cells
    FROM read_parquet('{rel}/dim_content.parquet')
""").df()

display(res_fact)
display(res_dim)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,tbl,total_rows,total_cells,total_cells_1,total_cells_2,total_cells_3,total_cells_4,total_cells_5,total_cells_6,total_cells_7,...,total_cells_21,total_cells_22,total_cells_23,total_cells_24,total_cells_25,total_cells_26,total_cells_27,total_cells_28,total_cells_29,total_cells_30
0,fact_content_daily_performance,78835655,78835655,78835655,78835655,78835655,78835655,78737649,49200328,78737649,...,49200328,49200328,49200328,49200328,49200328,49200328,49200328,49200328,49200328,78835655


,tbl,total_rows,total_cells,total_cells_1,total_cells_2,total_cells_3,total_cells_4,total_cells_5,total_cells_6,total_cells_7,...,total_cells_16,total_cells_17,total_cells_18,total_cells_19,total_cells_20,total_cells_21,total_cells_22,total_cells_23,total_cells_24,total_cells_25
0,dim_content,519606,519606,519606,447608,513081,519606,519606,519606,519606,...,519606,447608,149670,434643,341838,341838,45396,45396,519606,519606


## 3. Verify it with queries (grain, counts, missing values, windows)

Below, we write queries verifying:
1. The exact row count of `dim_content` (519,606 rows).
2. The unique key constraint verification for `dim_content` on `(client_hash_id, content_hash_id)`.
3. The existence of missing values across key metric fields in both datasets.

In [15]:
# 1. Verify Grain of dim_content (client_hash_id, content_hash_id)
grain_dim = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT (client_hash_id || '-' || content_hash_id)) as unique_keys
    FROM read_parquet('{rel}/dim_content.parquet')
""").df()
print("--- Dimension Table Grain Validation ---")
display(grain_dim)

# 2. Check for missing values in critical dimension features
null_checks = con.sql(f"""
    SELECT
        SUM(CASE WHEN client_hash_id IS NULL THEN 1 ELSE 0 END) as missing_client,
        SUM(CASE WHEN content_hash_id IS NULL THEN 1 ELSE 0 END) as missing_content,
        SUM(CASE WHEN search_volume IS NULL THEN 1 ELSE 0 END) as missing_search_volume,
        SUM(CASE WHEN model_used IS NULL THEN 1 ELSE 0 END) as missing_model
    FROM read_parquet('{rel}/dim_content.parquet')
""").df()
print("\n--- Dimension Null Value Checks ---")
display(null_checks)

--- Dimension Table Grain Validation ---


,total_rows,unique_keys
0,519606,519606



--- Dimension Null Value Checks ---


,missing_client,missing_content,missing_search_volume,missing_model
0,0.0,0.0,142622.0,84963.0


## 4. Data limits

Every dataset has boundaries and limits. Based on our verification, we observe the following boundaries:

1. **Unbalanced Client History / Late Content Onboarding**:
   * Not all content pages have performance data spanning the entire date range. Many content pages are created midway, meaning historical baselines are empty (or unbalanced) before their `content_created_date`.
2. **Missing Search Engine/AI Attribution**:
   * A significant chunk of content has missing AI metadata (e.g., `model_used` is null for **84,963** records) and missing search metrics (`search_volume` is null for **142,622** records), which represents non-AI generated legacy content or unindexed search queries.
3. **Granularity Limits**:
   * The metrics do not provide user-level pathways or real-time interactive actions—only aggregated, end-of-day summary metrics per content page.

In [16]:
# Let's check the percentage of content with missing AI Model attribution in dim_content
missing_stats = con.sql(f"""
    SELECT
        COUNT(*) as total_records,
        SUM(CASE WHEN model_used IS NULL THEN 1 ELSE 0 END) as missing_model_count,
        ROUND(100.0 * SUM(CASE WHEN model_used IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) as pct_missing_model,
        SUM(CASE WHEN search_volume IS NULL THEN 1 ELSE 0 END) as missing_search_vol_count,
        ROUND(100.0 * SUM(CASE WHEN search_volume IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) as pct_missing_search_vol
    FROM read_parquet('{rel}/dim_content.parquet')
""").df()

display(missing_stats)

,total_records,missing_model_count,pct_missing_model,missing_search_vol_count,pct_missing_search_vol
0,519606,84963.0,16.35,142622.0,27.45


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.